# 06 - Churn Event Analysis

This notebook distinguishes churn event rows from unique churned accounts and applies conservative subscription MRR attribution. Reactivation events are treated as reactivation evidence only; they are not churn events.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

from src.churn_metrics import load_sources, map_churn_events, monthly_churn_summary

In [ ]:
events, subscriptions = load_sources()
mapped = map_churn_events(events, subscriptions)
monthly = monthly_churn_summary(mapped)

print("Recorded event rows:", len(mapped))
print("Unique accounts with candidate churn events:",
      mapped.loc[~mapped["is_reactivation"], "account_id"].nunique())
print("Sequence-supported reactivations:",
      mapped["sequence_supported_reactivation"].sum())
monthly

In [ ]:
mapped[[
    "churn_event_id", "account_id", "churn_date", "is_reactivation",
    "sequence_assessment", "mapping_status", "candidate_subscription_id",
    "candidate_subscription_mrr", "other_not_ended_subscription_count",
    "candidate_account_churned_mrr"
]].head(20)

## Revenue interpretation

`candidate_account_churned_mrr` is populated only for a one-to-one same-account/end-date match with no other subscription unended at the event date. It is a conservative departure-candidate measure, not an MRR balance or a refund-adjusted revenue figure.